In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import mkdir
import json
from PIL import Image
from pycocotools import mask as mask_utils
from tqdm import tqdm
from mtrain.disk import DiskBooleanMask, DiskImage
import itertools

In [3]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
TRASH = NEG_MASKING_V1 / "trash"
SAMPLES_MAPILLARY = NEG_MASKING_V1 / "samples_mapillary"

CLIP_FILE_NAMES = ["clip_bottles.txt", "clip_litter.txt", "clip_plastic.txt", "clip_tobacco_packs.txt", "clip_delhi_litter.txt"]
CLIP_FILES = [TRASH / c for c in CLIP_FILE_NAMES]
print("all clip files exist:", all([f.exists() for f in CLIP_FILES]))

TRASH_DATA_DIR = TRASH / "data"

SAMPLES_MAPILLARY.exists(), TRASH.exists()

all clip files exist: True


(True, True)

In [4]:
from mtrain.neg_mask.widget_2 import LabelWidget, Bbox, parse_crop_level_ds, parse_top_level_ds

In [5]:
def get_region_crops(img, mask, padding=20):
    _, labels = cv2.connectedComponents(mask)
    h, w = img.shape[:2]
    for label in range(1, labels.max() + 1):
        rows, cols = np.where(labels == label)
        r1 = max(0, rows.min())
        r2 = min(h, rows.max())
        c1 = max(0, cols.min())
        c2 = min(w, cols.max())
        yield Bbox(c1, r1, c2-c1, r2-r1)

In [6]:
def get_crops(dirs):
    for d in dirs:
        img = DiskImage.load(d / "image.jpg")
        mask = DiskBooleanMask.load(d / "m2.png")
        for _, im, msk in get_region_crops(img, mask, 50):
            yield d.name, im, msk

In [7]:
# import os
# import shutil


# def mk_symlink(p):
#     os.symlink(
#         f"../../samples_mapillary/100/{p.name}", p, target_is_directory=True
#     )

# data = TRASH_DATA_DIR

# shutil.rmtree(data)
# data = mkdir(data)
# for d in dirs:
#     p = data / d.name
#     mk_symlink(p)

In [8]:
CLIP_FILES

[PosixPath('../../datasets/test-samples/neg-masking/V1/trash/clip_bottles.txt'),
 PosixPath('../../datasets/test-samples/neg-masking/V1/trash/clip_litter.txt'),
 PosixPath('../../datasets/test-samples/neg-masking/V1/trash/clip_plastic.txt'),
 PosixPath('../../datasets/test-samples/neg-masking/V1/trash/clip_tobacco_packs.txt'),
 PosixPath('../../datasets/test-samples/neg-masking/V1/trash/clip_delhi_litter.txt')]

In [35]:
def read_clip_file(path) -> list[tuple[str, Path]]:
    with open(path) as f:
        lines = f.readlines()
    imgs = [Path(line.split("\t")[1].strip()) for line in lines]
    dirs = [(path.stem, img.parent) for img in imgs]
    return dirs


def get_all_from_dir(path) -> list[tuple[str, Path]]:
    return [(path.stem, d) for d in path.glob("*")]

def get_dir_and_cat_iter():
    all_cat_dirs = [read_clip_file(f) for f in CLIP_FILES]
    # all_cat_dirs.append(get_all_from_dir(TRASH / "personal"))
    return itertools.chain.from_iterable(itertools.zip_longest(*all_cat_dirs))

def get_dir_for_widget():
    return (d for (_, d) in get_dir_and_cat_iter())

In [76]:
ROCKS = NEG_MASKING_V1 / "rocks"
# classification folder is here
widget = LabelWidget(ROCKS / "classification", crop_pad=220)

def is_dir_not_done(direc: Path):
    return not widget.is_done(direc.name)

dir_it = filter(is_dir_not_done, iter(get_dir_for_widget()))

# run the cell below for vscode color theme support in ipywidgets

# Source - https://stackoverflow.com/a/77028015
# Posted by Yingding Wang, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-02, License - CC BY-SA 4.0

In [82]:
d = next(dir_it)
img, mask = DiskImage.load(d / "image.jpg"), DiskBooleanMask.load(d / "m2.png")
bboxes = list(get_region_crops(img, mask))

widget.ui(d.name, bboxes, img,  mask)